## Y_all = mascara com valores de chuva (chuva = 0 é qnd nao tem a estaçao)
## M_all = máscara dizendo onde tem estação

#### pegando valor de chuva das estaçoes pra data do verao 2023-2024

In [2]:
import pandas as pd
from pathlib import Path

base_dir = Path("../../data/ws/full")

start = pd.Timestamp("2023-12-22", tz="UTC")
end = pd.Timestamp("2024-03-20 23:59:59", tz="UTC")

mapping_df = pd.read_csv("../src/datasets/mapeamento_pixel_estacao.csv")

all_rain = []

for station_id in range(1, 84):
    for year in [2023, 2024]:
        path = base_dir / f"station_id={station_id}" / f"year={year}" / "data.parquet"

        if not path.exists():
            continue

        df = pd.read_parquet(path)

        df = df[["id", "nome", "latitude", "longitude", "observation_datetime", "m15"]].copy()

        df = df.rename(columns={
            "id": "station_id"
        })

        df["observation_datetime"] = pd.to_datetime(
            df["observation_datetime"],
            utc=True
        )

        # REMOVE NaNs
        df = df.dropna(subset=["m15"])

        # alinha para janelas de 15 minutos
        # Ex: 22:40 vira 22:30; 21:10 vira 21:00
        df["observation_datetime"] = (
            df["observation_datetime"]
            .dt.floor("15min")
        )

        df = df[
            (df["observation_datetime"] >= start) &
            (df["observation_datetime"] <= end)
        ]

        if len(df) > 0:
            all_rain.append(df)

rain_df = pd.concat(all_rain, ignore_index=True)

# se após o floor existirem leituras repetidas, preservamos o maior valor de m15.
rain_df = rain_df.groupby(
    [
        "station_id",
        "nome",
        "latitude",
        "longitude",
        "observation_datetime"
    ],
    as_index=False
)["m15"].max()

rain_df = rain_df.merge(
    mapping_df[["station_id", "pixel_i", "pixel_j"]],
    on="station_id",
    how="inner"
)

print(rain_df.head())
print(rain_df.shape)
print(rain_df["observation_datetime"].min())
print(rain_df["observation_datetime"].max())
print("Estações:", rain_df["station_id"].nunique())

   station_id     nome  latitude  longitude      observation_datetime  m15  \
0           1  Adeus 1  -22.8636   -43.2636 2023-12-22 00:00:00+00:00  0.0   
1           1  Adeus 1  -22.8636   -43.2636 2023-12-22 00:15:00+00:00  0.0   
2           1  Adeus 1  -22.8636   -43.2636 2023-12-22 00:30:00+00:00  0.0   
3           1  Adeus 1  -22.8636   -43.2636 2023-12-22 00:45:00+00:00  0.0   
4           1  Adeus 1  -22.8636   -43.2636 2023-12-22 01:00:00+00:00  0.0   

   pixel_i  pixel_j  
0      303      324  
1      303      324  
2      303      324  
3      303      324  
4      303      324  
(692136, 8)
2023-12-22 00:00:00+00:00
2024-03-20 23:45:00+00:00
Estações: 83


In [3]:
print("\nMinutos presentes depois do alinhamento:")
print(rain_df["observation_datetime"].dt.minute.value_counts().sort_index())


Minutos presentes depois do alinhamento:
observation_datetime
0     173332
15    171446
30    173742
45    173616
Name: count, dtype: int64


In [4]:
timestamps = sorted(rain_df["observation_datetime"].unique())

print("Quantidade de timestamps:", len(timestamps))

print("Primeiro:", timestamps[0])
print("Último:", timestamps[-1])

Quantidade de timestamps: 8640
Primeiro: 2023-12-22 00:00:00+00:00
Último: 2024-03-20 23:45:00+00:00


## criando as mascaras

In [ ]:
## necessario se for trabalhar com resoluçao menor q a original, por causa da memoria 
H_orig = 656
W_orig = 654

H = 256
W = 256

rain_df["pixel_i_256"] = (
    rain_df["pixel_i"] * H / H_orig
).round().astype(int)

rain_df["pixel_j_256"] = (
    rain_df["pixel_j"] * W / W_orig
).round().astype(int)

rain_df["pixel_i_256"] = rain_df["pixel_i_256"].clip(0, H - 1)
rain_df["pixel_j_256"] = rain_df["pixel_j_256"].clip(0, W - 1)

In [7]:
import numpy as np

timestamps = sorted(rain_df["observation_datetime"].unique())

Y_all = np.zeros((len(timestamps), H, W, 1), dtype=np.float32)
M_all = np.zeros((len(timestamps), H, W, 1), dtype=np.uint8)

timestamp_to_idx = {ts: idx for idx, ts in enumerate(timestamps)}

for _, row in rain_df.iterrows():
    t_idx = timestamp_to_idx[row["observation_datetime"]]

    i = int(row["pixel_i_256"])
    j = int(row["pixel_j_256"])

    rain = float(row["m15"])

    Y_all[t_idx, i, j, 0] = max(
        Y_all[t_idx, i, j, 0],
        rain
    )

    M_all[t_idx, i, j, 0] = 1

print("Y_all:", Y_all.shape)
print("M_all:", M_all.shape)
print("timestamps:", len(timestamps))
print("observações:", M_all.sum())

Y_all: (8640, 256, 256, 1)
M_all: (8640, 256, 256, 1)
timestamps: 8640
observações: 497421


In [21]:
print(rain_df[["pixel_i", "pixel_j", "pixel_i_256", "pixel_j_256"]].head())
print(rain_df["pixel_i_256"].min(), rain_df["pixel_i_256"].max())
print(rain_df["pixel_j_256"].min(), rain_df["pixel_j_256"].max())

   pixel_i  pixel_j  pixel_i_256  pixel_j_256
0      303      324          118          127
1      303      324          118          127
2      303      324          118          127
3      303      324          118          127
4      303      324          118          127
115 132
118 136


In [22]:
print("NaNs em m15:", rain_df["m15"].isna().sum())

NaNs em m15: 0


In [23]:
print("estações médias por timestamp:", M_all.sum() / len(timestamps))
print("chuva máxima:", Y_all.max())
print("chuva > 0:", (Y_all[M_all == 1] > 0).sum())

estações médias por timestamp: 57.571875
chuva máxima: 60.0
chuva > 0: 28815


In [24]:
print(
    rain_df.sort_values("m15", ascending=False)[
        [
            "station_id",
            "nome",
            "observation_datetime",
            "m15",
            "pixel_i_256",
            "pixel_j_256"
        ]
    ].head(20)
)

        station_id                                 nome  \
174654          22                    EspÃ­rito Santo 1   
78628           10                          Cantagalo 1   
78629           10                          Cantagalo 1   
78964           10                          Cantagalo 1   
235934          29                    Jardim do Carmo 1   
78627           10                          Cantagalo 1   
174653          22                    EspÃ­rito Santo 1   
686731          83              Vila Pereira da Silva 1   
78963           10                          Cantagalo 1   
184122          23                     Fazenda Catete 1   
235935          29                    Jardim do Carmo 1   
459343          56                           Prazeres 1   
79322           10                          Cantagalo 1   
551162          67  Santos Rodrigues 2 / Azevedo Lima 1   
513350          62              Rua BrÃ­cio de Moraes 1   
475747          58                     Rio das Pedras 2 

In [25]:
print(rain_df["m15"].describe())
print(rain_df.sort_values("m15", ascending=False).head(20))

count    692136.000000
mean          0.055119
std           0.550359
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          60.000000
Name: m15, dtype: float64
        station_id                                 nome  latitude  longitude  \
174654          22                    EspÃ­rito Santo 1  -22.8900   -43.3439   
78628           10                          Cantagalo 1  -22.9808   -43.1969   
78629           10                          Cantagalo 1  -22.9808   -43.1969   
78964           10                          Cantagalo 1  -22.9808   -43.1969   
235934          29                    Jardim do Carmo 1  -22.8514   -43.3045   
78627           10                          Cantagalo 1  -22.9808   -43.1969   
174653          22                    EspÃ­rito Santo 1  -22.8900   -43.3439   
686731          83              Vila Pereira da Silva 1  -22.9307   -43.1914   
78963           10                          Cantagalo 1  -22.9808   -4

In [8]:
data = np.load(
    "/home/noemi/atmoseer/data/datasets/dataset_radar_verao_2023_2024_15min_256_5in_5out_stride5.npz",
    allow_pickle=True
)

X = data["X"]
radar_y_timestamps = data["y_timestamps"]

print("X:", X.shape)
print("radar_y_timestamps:", radar_y_timestamps.shape)
print("primeiro:", radar_y_timestamps[0])

X: (1449, 5, 256, 256, 3)
radar_y_timestamps: (1449, 5)
primeiro: ['2023-12-22T01:15:00' '2023-12-22T01:30:00' '2023-12-22T01:45:00'
 '2023-12-22T02:00:00' '2023-12-22T02:15:00']


In [27]:
print(data.files)

['X', 'Y', 'y_timestamps', 'metadata']


In [28]:
def normalize_ts(ts):
    ts = pd.Timestamp(ts)

    if ts.tzinfo is None:
        ts = ts.tz_localize("UTC")
    else:
        ts = ts.tz_convert("UTC")

    return ts

timestamp_to_idx = {
    normalize_ts(ts): idx
    for idx, ts in enumerate(timestamps)
}

In [29]:
Y_seq = []
M_seq = []
X_valid = []

missing = 0

for sample_idx, sample_ts in enumerate(radar_y_timestamps):
    y_frames = []
    m_frames = []

    ok = True

    for ts in sample_ts:
        ts_norm = normalize_ts(ts)

        if ts_norm not in timestamp_to_idx:
            missing += 1
            ok = False
            break

        idx = timestamp_to_idx[ts_norm]

        y_frames.append(Y_all[idx])
        m_frames.append(M_all[idx])

    if ok:
        Y_seq.append(np.stack(y_frames, axis=0))
        M_seq.append(np.stack(m_frames, axis=0))
        X_valid.append(X[sample_idx])

Y_seq = np.array(Y_seq, dtype=np.float32)
M_seq = np.array(M_seq, dtype=np.uint8)
X_valid = np.array(X_valid, dtype=X.dtype)

print("X original:", X.shape)
print("X_valid:", X_valid.shape)
print("Y_seq:", Y_seq.shape)
print("M_seq:", M_seq.shape)
print("missing:", missing)

X original: (1449, 5, 256, 256, 3)
X_valid: (1449, 5, 256, 256, 3)
Y_seq: (1449, 5, 256, 256, 1)
M_seq: (1449, 5, 256, 256, 1)
missing: 0


In [30]:
print(X.shape[0])
print(Y_seq.shape[0])

1449
1449


In [31]:
np.savez_compressed(
    "/home/noemi/atmoseer/data/datasets/dataset_radar_ws_masked_15min_256_5in_5out_stride5.npz",
    X=X_valid,
    Y=Y_seq,
    M=M_seq,
    y_timestamps=radar_y_timestamps,
)

In [32]:
bins = [0, 1.25, 6.25, 12.5, np.inf]

labels = [
    "Fraca (<1.25 mm/15min)",
    "Moderada (1.25-6.25 mm/15min)",
    "Forte (6.25-12.5 mm/15min)",
    "Extrema (>12.5 mm/15min)"
]

# pega apenas pixels com estação
rain_values = Y_seq[M_seq == 1]

classes = pd.cut(
    rain_values,
    bins=bins,
    labels=labels,
    include_lowest=True
)

counts = classes.value_counts().sort_index()

percent = (
    counts / counts.sum() * 100
).round(2)

summary = pd.DataFrame({
    "count": counts,
    "percent": percent
})

print(summary)

                                count  percent
Fraca (<1.25 mm/15min)         412020    98.67
Moderada (1.25-6.25 mm/15min)    4860     1.16
Forte (6.25-12.5 mm/15min)        467     0.11
Extrema (>12.5 mm/15min)          235     0.06


In [33]:
# pega apenas pixels com estação
rain_values = Y_seq[M_seq == 1]

# remove ausência de chuva
rain_values = rain_values[rain_values > 0]

bins = [0, 1.25, 6.25, 12.5, np.inf]

labels = [
    "Fraca (<1.25 mm/15min)",
    "Moderada (1.25-6.25 mm/15min)",
    "Forte (6.25-12.5 mm/15min)",
    "Extrema (>12.5 mm/15min)"
]

classes = pd.cut(
    rain_values,
    bins=bins,
    labels=labels,
    include_lowest=True
)

counts = classes.value_counts().sort_index()

percent = (
    counts / counts.sum() * 100
).round(2)

summary = pd.DataFrame({
    "count": counts,
    "percent": percent
})

print(summary)

                               count  percent
Fraca (<1.25 mm/15min)         23098    80.59
Moderada (1.25-6.25 mm/15min)   4860    16.96
Forte (6.25-12.5 mm/15min)       467     1.63
Extrema (>12.5 mm/15min)         235     0.82
